# **Pendahuluan**

## Import Library

In [1]:
import pandas as pd
import numpy as np
import re
import ast
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sentence_transformers import SentenceTransformer

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Dataset

Baca dataset cv dan job

In [3]:
df_cv = pd.read_csv('/content/drive/MyDrive/STUPEN/Capstone - Checkpoint 2/sample_100_cv_with_ground_truth.csv')
df_job = pd.read_csv('/content/drive/MyDrive/STUPEN/Capstone - Checkpoint 2/data_clean.csv')

Tampilkan dataset cv

In [4]:
df_cv.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"['Perawat', 'Nurse', 'Medical Representative']"
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","['Perawat', 'Nurse', 'Medical Representative']"
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"['Customer Service', 'Customer Success Officer..."
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",['Data Analyst for Production Staff (CODE : PR...
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"['Engineer Staff', 'Project Engineer', 'Electr..."


Ubah setiap baris pada kolom "ground_truth" menjadi list (hanya dijalankan jika df_cv adalah sample_10_cv.csv)

In [5]:
# df_cv["ground_truth"] = df_cv["ground_truth"].apply(
#     lambda x: [item.strip() for item in x.split(",")]
# )
# df_cv['ground_truth'].head()

Ubah setiap baris pada kolom "ground_truth" menjadi list (hanya dijalankan jika df_cv adalah sample_100_cv.csv)

In [6]:
df_cv["ground_truth"] = df_cv["ground_truth"].apply(ast.literal_eval)
df_cv["ground_truth"].head()

,ground_truth
0,"[Perawat, Nurse, Medical Representative]"
1,"[Perawat, Nurse, Medical Representative]"
2,"[Customer Service, Customer Success Officer, C..."
3,[Data Analyst for Production Staff (CODE : PRO...
4,"[Engineer Staff, Project Engineer, Electrical ..."


Tampilkan dataset job

In [7]:
df_job.head()

,Posisi,Perusahaan,Lokasi,Type,Gaji,Requirements,Link,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation
0,Tenaga Admin,PT Serdang Baja Makmur Abadi,Medan,Full time,Rp 3.250.000 – Rp 3.500.000 per month,Pekerjaan administrasi umum seperti membuat In...,https://id.jobstreet.com/id/job/91338594?type=...,admin staff admin staff general administrative...,admin staff general administrative work such a...,admin staff general administrative work such a...
1,Admin Operasional & HR Support (MAGANG),PT Saptakarsa Prima,Kecamatan Tangerang,Full time,Rp 2.500.000 – Rp 3.500.000 per month,Pendidikan minimal SMK Jurusan Administrasi Pe...,https://id.jobstreet.com/id/job/91309622?type=...,admin operational hr support internship admin ...,admin operational hr support internship educat...,admin operational hr support internship educat...
2,Executive admin,Pengiklan Anonim,Jakarta Utara,Full time,Rp 4.700.000 – Rp 5.500.000 per month,Mengelola kalender dan jadwal eksekutif dengan...,https://id.jobstreet.com/id/job/91479903?type=...,executive admin executive admin manages execut...,executive admin manages executive calendars an...,executive admin manages executive calendars an...
3,Executive admin,Pengiklan Anonim,Jakarta Utara,Full time,Rp 4.700.000 – Rp 5.500.000 per month,"Mengelola jadwal, pertemuan, dan kegiatan ekse...",https://id.jobstreet.com/id/job/91508884?type=...,executive admin executive admin manages meetin...,executive admin manages meeting schedules and ...,executive admin manages meeting schedules and ...
4,Operations & Admin Coordinator (Indonesia Repr...,Pengiklan Anonim,Cilandak,Full time,Rp 6.500.000 – Rp 8.500.000 per month,Manage daily administrative tasks; Organize do...,https://id.jobstreet.com/id/job/91487019?type=...,operations admin coordinator indonesia represe...,operations admin coordinator indonesia represe...,operations admin coordinator indonesia represe...


Informasi dataset cv

In [8]:
df_cv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   candidate_name                   100 non-null    object
 1   email                            100 non-null    object
 2   phone                            100 non-null    int64 
 3   skills                           100 non-null    object
 4   summary                          100 non-null    object
 5   experience                       100 non-null    object
 6   degree                           100 non-null    object
 7   university                       100 non-null    object
 8   Category                         100 non-null    object
 9   text_for_tfidf                   100 non-null    object
 10  text_for_embed                   100 non-null    object
 11  text_for_embed_with_punctuation  100 non-null    object
 12  ground_truth                     100 

Informasi dataset job

In [9]:
df_job.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1400 entries, 0 to 1399
Data columns (total 10 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Posisi                           1400 non-null   object
 1   Perusahaan                       1400 non-null   object
 2   Lokasi                           1400 non-null   object
 3   Type                             1400 non-null   object
 4   Gaji                             1400 non-null   object
 5   Requirements                     1400 non-null   object
 6   Link                             1400 non-null   object
 7   text_for_tfidf                   1400 non-null   object
 8   text_for_embed                   1400 non-null   object
 9   text_for_embed_with_punctuation  1400 non-null   object
dtypes: object(10)
memory usage: 109.5+ KB


In [10]:
# df_cv = df_cv.sample(n=75, random_state=42).reset_index(drop=True)

In [11]:
# df_cv_clean = df_cv.drop_duplicates(subset=['candidate_name'])
# df_cv_undersampling = df_cv_clean.sample(n=100, random_state=42).reset_index(drop=True)
# df_cv_undersampling

In [12]:
# df_cv_undersampling.to_csv('sample_100_cv.csv', index=False)

In [13]:
# df_job.iloc[:,0].tolist()

# **TF-IDF**

## Model

Kolom "text_for_tfidf" dari dataset job dijadikan list

In [14]:
job_texts = df_job["text_for_tfidf"].tolist()

Tuning TF-IDF

In [15]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    sublinear_tf=True
)

job_tfidf = vectorizer.fit_transform(job_texts)

Mulai prediksi berdasarkan kolom "text_for_tfidf" dari **df_cv** dan **df_job**

In [16]:
prediksi_perusahaan = []
prediksi_posisi = []
hasil_cosine_similarity = []

for h in range(len(df_cv)):
    prediksi_perusahaan_tiap_baris = []
    prediksi_posisi_tiap_baris = []
    hasil_cosine_similarity_tiap_baris = []

    # print("-"*40)
    # print(f"BARIS {h+1}")
    # print("-"*40)
    cv_text = df_cv.loc[h, 'text_for_tfidf']
    cv_tfidf = vectorizer.transform([cv_text])

    scores = cosine_similarity(cv_tfidf, job_tfidf)

    top_k = 5
    top_indices = scores[0].argsort()[-top_k:][::-1]

    for i in top_indices:
        # print("Perusahaan:", df_job.iloc[i]["Perusahaan"])
        # print("Posisi:", df_job.iloc[i]["Posisi"])
        # print("Score:", scores[0][i])
        # print("-"*40)
        prediksi_perusahaan_tiap_baris.append(df_job.iloc[i]["Perusahaan"])
        prediksi_posisi_tiap_baris.append(df_job.iloc[i]["Posisi"])
        hasil_cosine_similarity_tiap_baris.append(float(scores[0][i]))

    prediksi_perusahaan.append(prediksi_perusahaan_tiap_baris)
    prediksi_posisi.append(prediksi_posisi_tiap_baris)
    hasil_cosine_similarity.append(hasil_cosine_similarity_tiap_baris)

    # print("\n")

Membuat salinan dari **df_cv** dan menambahkan beberapa kolom yang berisi hasil prediksi

In [17]:
df_cv_hasil_tfidf = df_cv.copy()
df_cv_hasil_tfidf['prediksi_perusahaan'] = prediksi_perusahaan
df_cv_hasil_tfidf['prediksi_posisi'] = prediksi_posisi
df_cv_hasil_tfidf['cosine_similarity'] = hasil_cosine_similarity
df_cv_hasil_tfidf.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth,prediksi_perusahaan,prediksi_posisi,cosine_similarity
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"[Perawat, Nurse, Medical Representative]","[Serrebeauty, RS Awal Bros Group, Pengiklan An...","[Dokter Umum, Kepala Departemen Penunjang Medi...","[0.2805980835611086, 0.27447319106128015, 0.27..."
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","[Perawat, Nurse, Medical Representative]","[Pengiklan Anonim, RS Firdaus, Klinik Sakti Me...","[Private Homecare Nurse, Perawat, Perawat Beda...","[0.3616282862230051, 0.25756608059456326, 0.19..."
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"[Customer Service, Customer Success Officer, C...","[PT ASURANSI SIMAS JIWA, Nusa Medica Clinic, P...","[Customer Service Sharia (Contract), Call Cent...","[0.25884641051911794, 0.25250789015872643, 0.2..."
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",[Data Analyst for Production Staff (CODE : PRO...,"[PT Waterpro Mandiri International, OPPO Indon...","[Process Engineer (WTP & WWTP), Industrial Eng...","[0.19294035712359686, 0.18777350672419194, 0.1..."
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"[Engineer Staff, Project Engineer, Electrical ...",[PT. Sarana Meditama International (EMC Health...,"[AI Application Officer, IT, Electrical Engine...","[0.1772868244496228, 0.1685384463276526, 0.132..."


## Evaluasi

Menyiapkan preprocessing sederhana dan model pre-train untuk evaluasi

In [18]:
def normalize(text):
    text = text.lower()
    text = re.sub(r'\([^)]*\)', '', text)  # hapus isi kurung
    return text.strip()

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Menghitung nilai cosine similarity maksimum antara **ground truth** dan **hasil prediksi** pada setiap data, kemudian menentukan **threshold similarity** berdasarkan nilai kuartil ketiga (Q3) dari distribusi similarity maksimum yang diperoleh.

In [19]:
nilai_sim_max = []

for i in range(len(df_cv_hasil_tfidf)):

    ground_truth = [normalize(x) for x in df_cv_hasil_tfidf.loc[i,'ground_truth']]
    hasil_prediksi = [normalize(x) for x in df_cv_hasil_tfidf.loc[i,'prediksi_posisi']]

    sim = cosine_similarity(
        model.encode(ground_truth),
        model.encode(hasil_prediksi)
    )
    nilai_sim_max.extend(sim.max(axis=0))

In [20]:
series_sim_max = pd.Series(nilai_sim_max)
q3 = series_sim_max.quantile(0.75)
threshold = np.floor(q3 * 10) / 10
print(f'q3 = {q3}')
print(f'threshold = {threshold}')

q3 = 0.7211241275072098
threshold = 0.7


Menentukan relevansi setiap hasil prediksi berdasarkan nilai cosine similarity terhadap **ground truth** menggunakan threshold yang telah diperoleh, kemudian menghitung nilai **Precision@5** untuk mengukur proporsi prediksi yang relevan pada setiap data.

In [21]:
relevan_atau_tidak = []
nilai_precision_at_5 = []

for i in range(len(df_cv_hasil_tfidf)):

    ground_truth = [normalize(x) for x in df_cv_hasil_tfidf.loc[i,'ground_truth']]
    hasil_prediksi = [normalize(x) for x in df_cv_hasil_tfidf.loc[i,'prediksi_posisi']]

    sim = cosine_similarity(
        model.encode(ground_truth),
        model.encode(hasil_prediksi)
    )

    relevan = (sim.max(axis=0) >= threshold).astype(int).tolist()
    precision_at_5 = sum(relevan) / len(relevan)

    relevan_atau_tidak.append(relevan)
    nilai_precision_at_5.append(precision_at_5)

df_cv_hasil_tfidf['relevan/tidak'] = relevan_atau_tidak
df_cv_hasil_tfidf['precision@5'] = nilai_precision_at_5
df_cv_hasil_tfidf.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth,prediksi_perusahaan,prediksi_posisi,cosine_similarity,relevan/tidak,precision@5
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"[Perawat, Nurse, Medical Representative]","[Serrebeauty, RS Awal Bros Group, Pengiklan An...","[Dokter Umum, Kepala Departemen Penunjang Medi...","[0.2805980835611086, 0.27447319106128015, 0.27...","[1, 1, 1, 1, 1]",1.0
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","[Perawat, Nurse, Medical Representative]","[Pengiklan Anonim, RS Firdaus, Klinik Sakti Me...","[Private Homecare Nurse, Perawat, Perawat Beda...","[0.3616282862230051, 0.25756608059456326, 0.19...","[1, 1, 1, 1, 0]",0.8
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"[Customer Service, Customer Success Officer, C...","[PT ASURANSI SIMAS JIWA, Nusa Medica Clinic, P...","[Customer Service Sharia (Contract), Call Cent...","[0.25884641051911794, 0.25250789015872643, 0.2...","[0, 0, 0, 1, 1]",0.4
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",[Data Analyst for Production Staff (CODE : PRO...,"[PT Waterpro Mandiri International, OPPO Indon...","[Process Engineer (WTP & WWTP), Industrial Eng...","[0.19294035712359686, 0.18777350672419194, 0.1...","[0, 0, 0, 0, 0]",0.0
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"[Engineer Staff, Project Engineer, Electrical ...",[PT. Sarana Meditama International (EMC Health...,"[AI Application Officer, IT, Electrical Engine...","[0.1772868244496228, 0.1685384463276526, 0.132...","[0, 0, 1, 0, 1]",0.4


Hitung rata-rata precision@5

In [22]:
mean_precision_at_5_tfidf = df_cv_hasil_tfidf['precision@5'].mean()
print(f"Nilai rata-rata precision@5: {mean_precision_at_5_tfidf}")

Nilai rata-rata precision@5: 0.28


# **TF-IDF + SVD**

## Model

Kolom "text_for_tfidf" dari dataset job dijadikan list

In [23]:
job_texts = df_job["text_for_tfidf"].tolist()

Tuning TF-IDF

In [24]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    sublinear_tf=True
)

job_tfidf = vectorizer.fit_transform(job_texts)

Tuning SVD

In [25]:
svd = TruncatedSVD(n_components=100, random_state=42)

job_svd = svd.fit_transform(job_tfidf)

Mulai prediksi berdasarkan kolom "text_for_tfidf" dari **df_cv** dan **df_job**

In [26]:
prediksi_perusahaan = []
prediksi_posisi = []
hasil_cosine_similarity = []

for h in range(len(df_cv)):
    prediksi_perusahaan_tiap_baris = []
    prediksi_posisi_tiap_baris = []
    hasil_cosine_similarity_tiap_baris = []

    # print("-"*40)
    # print(f"BARIS {h+1}")
    # print("-"*40)
    cv_text = df_cv.loc[h, 'text_for_tfidf']
    cv_tfidf = vectorizer.transform([cv_text])
    cv_svd = svd.transform(cv_tfidf)

    scores_svd = cosine_similarity(cv_svd, job_svd)

    top_k = 5
    top_indices = scores_svd[0].argsort()[-top_k:][::-1]

    for i in top_indices:
        # print("Perusahaan:", df_job.iloc[i]["Perusahaan"])
        # print("Posisi:", df_job.iloc[i]["Posisi"])
        # print("Score:", scores_svd[0][i])
        # print("-"*40)
        prediksi_perusahaan_tiap_baris.append(df_job.iloc[i]["Perusahaan"])
        prediksi_posisi_tiap_baris.append(df_job.iloc[i]["Posisi"])
        hasil_cosine_similarity_tiap_baris.append(float(scores_svd[0][i]))

    prediksi_perusahaan.append(prediksi_perusahaan_tiap_baris)
    prediksi_posisi.append(prediksi_posisi_tiap_baris)
    hasil_cosine_similarity.append(hasil_cosine_similarity_tiap_baris)

    # print("\n")

Membuat salinan dari **df_cv** dan menambahkan beberapa kolom yang berisi hasil prediksi

In [27]:
df_cv_hasil_tfidf_svd = df_cv.copy()
df_cv_hasil_tfidf_svd['prediksi_perusahaan'] = prediksi_perusahaan
df_cv_hasil_tfidf_svd['prediksi_posisi'] = prediksi_posisi
df_cv_hasil_tfidf_svd['cosine_similarity'] = hasil_cosine_similarity
df_cv_hasil_tfidf_svd.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth,prediksi_perusahaan,prediksi_posisi,cosine_similarity
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"[Perawat, Nurse, Medical Representative]","[PT GENOMIK SOLIDARITAS INDONESIA, Pengiklan A...","[Perawat Perusahaan, Private Homecare Nurse, D...","[0.7510799251188788, 0.7279133242296282, 0.722..."
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","[Perawat, Nurse, Medical Representative]","[Pengiklan Anonim, Klinik Sakti Medika, RS Fir...","[Private Homecare Nurse, Perawat Bedah, Perawa...","[0.7696247769620093, 0.7617207325170107, 0.750..."
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"[Customer Service, Customer Success Officer, C...","[Staffinc, Staffinc, PT Indotel Maju Bersama, ...","[Customer Service Yogyakarta, Customer Service...","[0.732876789662348, 0.7134074070189012, 0.6932..."
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",[Data Analyst for Production Staff (CODE : PRO...,"[PT Airtekindo Prima, OPPO Indonesia, PT Water...","[ENGINEERING HVAC SUPERVISOR, Industrial Engin...","[0.6053206028445351, 0.5725448484634272, 0.551..."
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"[Engineer Staff, Project Engineer, Electrical ...","[PT Marktel, PT. Sarana Meditama International...","[Programmer R&D, AI Application Officer, Presa...","[0.6248788478675723, 0.6195014220331998, 0.582..."


## Evaluasi

Menyiapkan preprocessing sederhana dan model pre-train untuk evaluasi

In [28]:
def normalize(text):
    text = text.lower()
    text = re.sub(r'\([^)]*\)', '', text)  # hapus isi kurung
    return text.strip()

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Menghitung nilai cosine similarity maksimum antara **ground truth** dan **hasil prediksi** pada setiap data, kemudian menentukan **threshold similarity** berdasarkan nilai kuartil ketiga (Q3) dari distribusi similarity maksimum yang diperoleh.

In [29]:
nilai_sim_max = []

for i in range(len(df_cv_hasil_tfidf_svd)):

    ground_truth = [normalize(x) for x in df_cv_hasil_tfidf_svd.loc[i,'ground_truth']]
    hasil_prediksi = [normalize(x) for x in df_cv_hasil_tfidf_svd.loc[i,'prediksi_posisi']]

    sim = cosine_similarity(
        model.encode(ground_truth),
        model.encode(hasil_prediksi)
    )
    nilai_sim_max.extend(sim.max(axis=0))

In [30]:
series_sim_max = pd.Series(nilai_sim_max)
q3 = series_sim_max.quantile(0.75)
threshold = np.floor(q3 * 10) / 10
print(f'q3 = {q3}')
print(f'threshold = {threshold}')

q3 = 0.7673467248678207
threshold = 0.7


Menentukan relevansi setiap hasil prediksi berdasarkan nilai cosine similarity terhadap **ground truth** menggunakan threshold yang telah diperoleh, kemudian menghitung nilai **Precision@5** untuk mengukur proporsi prediksi yang relevan pada setiap data.

In [31]:
relevan_atau_tidak = []
nilai_precision_at_5 = []

for i in range(len(df_cv_hasil_tfidf_svd)):

    ground_truth = [normalize(x) for x in df_cv_hasil_tfidf_svd.loc[i,'ground_truth']]
    hasil_prediksi = [normalize(x) for x in df_cv_hasil_tfidf_svd.loc[i,'prediksi_posisi']]

    sim = cosine_similarity(
        model.encode(ground_truth),
        model.encode(hasil_prediksi)
    )

    relevan = (sim.max(axis=0) >= threshold).astype(int).tolist()
    precision_at_5 = sum(relevan) / len(relevan)

    relevan_atau_tidak.append(relevan)
    nilai_precision_at_5.append(precision_at_5)

df_cv_hasil_tfidf_svd['relevan/tidak'] = relevan_atau_tidak
df_cv_hasil_tfidf_svd['precision@5'] = nilai_precision_at_5
df_cv_hasil_tfidf_svd.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth,prediksi_perusahaan,prediksi_posisi,cosine_similarity,relevan/tidak,precision@5
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"[Perawat, Nurse, Medical Representative]","[PT GENOMIK SOLIDARITAS INDONESIA, Pengiklan A...","[Perawat Perusahaan, Private Homecare Nurse, D...","[0.7510799251188788, 0.7279133242296282, 0.722...","[1, 1, 1, 1, 1]",1.0
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","[Perawat, Nurse, Medical Representative]","[Pengiklan Anonim, Klinik Sakti Medika, RS Fir...","[Private Homecare Nurse, Perawat Bedah, Perawa...","[0.7696247769620093, 0.7617207325170107, 0.750...","[1, 1, 1, 1, 1]",1.0
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"[Customer Service, Customer Success Officer, C...","[Staffinc, Staffinc, PT Indotel Maju Bersama, ...","[Customer Service Yogyakarta, Customer Service...","[0.732876789662348, 0.7134074070189012, 0.6932...","[1, 1, 0, 1, 1]",0.8
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",[Data Analyst for Production Staff (CODE : PRO...,"[PT Airtekindo Prima, OPPO Indonesia, PT Water...","[ENGINEERING HVAC SUPERVISOR, Industrial Engin...","[0.6053206028445351, 0.5725448484634272, 0.551...","[0, 0, 0, 1, 0]",0.2
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"[Engineer Staff, Project Engineer, Electrical ...","[PT Marktel, PT. Sarana Meditama International...","[Programmer R&D, AI Application Officer, Presa...","[0.6248788478675723, 0.6195014220331998, 0.582...","[0, 0, 0, 1, 0]",0.2


Hitung rata-rata precision@5

In [32]:
mean_precision_at_5_tfidf_svd = df_cv_hasil_tfidf_svd['precision@5'].mean()
print(f"Nilai rata-rata precision@5: {mean_precision_at_5_tfidf_svd}")

Nilai rata-rata precision@5: 0.33400000000000013


# **Embedding**

## Model

Kolom "text_for_embed" dari dataset job dijadikan list

In [33]:
job_texts = df_job["text_for_embed"].tolist()

Memanggil model embedding pre-train untuk prediksi

In [34]:
model = SentenceTransformer('all-MiniLM-L6-v2')
# model = SentenceTransformer('all-mpnet-base-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
job_embeddings = model.encode(job_texts, show_progress_bar=True)

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Mulai prediksi berdasarkan kolom "text_for_embed" dari **df_cv** dan **df_job**

In [36]:
prediksi_perusahaan = []
prediksi_posisi = []
hasil_cosine_similarity = []

for h in range(len(df_cv)):
    prediksi_perusahaan_tiap_baris = []
    prediksi_posisi_tiap_baris = []
    hasil_cosine_similarity_tiap_baris = []

    # print("-"*40)
    # print(f"BARIS {h+1}")
    # print("-"*40)
    cv_text = df_cv.loc[h, 'text_for_embed']
    cv_embedding = model.encode([cv_text])

    scores_embed = cosine_similarity(cv_embedding, job_embeddings)

    top_k = 5
    top_indices = scores_embed[0].argsort()[-top_k:][::-1]

    for i in top_indices:
        # print("Perusahaan:", df_job.iloc[i]["Perusahaan"])
        # print("Posisi:", df_job.iloc[i]["Posisi"])
        # print("Score:", scores_embed[0][i])
        # print("-"*40)
        prediksi_perusahaan_tiap_baris.append(df_job.iloc[i]["Perusahaan"])
        prediksi_posisi_tiap_baris.append(df_job.iloc[i]["Posisi"])
        hasil_cosine_similarity_tiap_baris.append(float(scores_embed[0][i]))

    prediksi_perusahaan.append(prediksi_perusahaan_tiap_baris)
    prediksi_posisi.append(prediksi_posisi_tiap_baris)
    hasil_cosine_similarity.append(hasil_cosine_similarity_tiap_baris)

    # print("\n")

Membuat salinan dari **df_cv** dan menambahkan beberapa kolom yang berisi hasil prediksi

In [37]:
df_cv_hasil_embedding = df_cv.copy()
df_cv_hasil_embedding['prediksi_perusahaan'] = prediksi_perusahaan
df_cv_hasil_embedding['prediksi_posisi'] = prediksi_posisi
df_cv_hasil_embedding['cosine_similarity'] = hasil_cosine_similarity
df_cv_hasil_embedding.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth,prediksi_perusahaan,prediksi_posisi,cosine_similarity
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"[Perawat, Nurse, Medical Representative]","[Serrebeauty, PT Transfarma Medica Indah, Nusa...","[Dokter Umum, Medical Representative (Surabaya...","[0.7163448333740234, 0.6829164028167725, 0.672..."
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","[Perawat, Nurse, Medical Representative]","[Klinik Sakti Medika, EUROMEDICA GROUP, PT RAD...","[Perawat Bedah, perawat, Nurse - Perawat, Nurs...","[0.5552957057952881, 0.5384441614151001, 0.527..."
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"[Customer Service, Customer Success Officer, C...","[PT Cilia Prisma Utama Makmur, PT Giant Transp...",[Sales Representative Tangerang / Jakarta / Be...,"[0.6683275699615479, 0.6631754636764526, 0.662..."
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",[Data Analyst for Production Staff (CODE : PRO...,"[PT Panasonic Gobel Energy Indonesia (PECGI), ...",[Data Analyst for Production Staff (CODE : PRO...,"[0.600621223449707, 0.5836132764816284, 0.5434..."
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"[Engineer Staff, Project Engineer, Electrical ...","[PT Derma Enom Farma, PT. Sarana Meditama Inte...","[Electrical Engineering (Kosmetik & Farmasi), ...","[0.6778924465179443, 0.6648625135421753, 0.628..."


## Evaluasi

Menyiapkan preprocessing sederhana dan model pre-train untuk evaluasi

In [38]:
def normalize(text):
    text = text.lower()
    text = re.sub(r'\([^)]*\)', '', text)  # hapus isi kurung
    return text.strip()

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Menghitung nilai cosine similarity maksimum antara **ground truth** dan **hasil prediksi** pada setiap data, kemudian menentukan **threshold similarity** berdasarkan nilai kuartil ketiga (Q3) dari distribusi similarity maksimum yang diperoleh.

In [39]:
nilai_sim_max = []

for i in range(len(df_cv_hasil_embedding)):

    ground_truth = [normalize(x) for x in df_cv_hasil_embedding.loc[i,'ground_truth']]
    hasil_prediksi = [normalize(x) for x in df_cv_hasil_embedding.loc[i,'prediksi_posisi']]

    sim = cosine_similarity(
        model.encode(ground_truth),
        model.encode(hasil_prediksi)
    )
    nilai_sim_max.extend(sim.max(axis=0))

In [40]:
series_sim_max = pd.Series(nilai_sim_max)
q3 = series_sim_max.quantile(0.75)
threshold = np.floor(q3 * 10) / 10
print(f'q3 = {q3}')
print(f'threshold = {threshold}')

q3 = 0.7460968494415283
threshold = 0.7


Menentukan relevansi setiap hasil prediksi berdasarkan nilai cosine similarity terhadap **ground truth** menggunakan threshold yang telah diperoleh, kemudian menghitung nilai **Precision@5** untuk mengukur proporsi prediksi yang relevan pada setiap data.

In [41]:
relevan_atau_tidak = []
nilai_precision_at_5 = []

for i in range(len(df_cv_hasil_embedding)):

    ground_truth = [normalize(x) for x in df_cv_hasil_embedding.loc[i,'ground_truth']]
    hasil_prediksi = [normalize(x) for x in df_cv_hasil_embedding.loc[i,'prediksi_posisi']]

    sim = cosine_similarity(
        model.encode(ground_truth),
        model.encode(hasil_prediksi)
    )

    relevan = (sim.max(axis=0) >= threshold).astype(int).tolist()
    precision_at_5 = sum(relevan) / len(relevan)

    relevan_atau_tidak.append(relevan)
    nilai_precision_at_5.append(precision_at_5)

df_cv_hasil_embedding['relevan/tidak'] = relevan_atau_tidak
df_cv_hasil_embedding['precision@5'] = nilai_precision_at_5
df_cv_hasil_embedding.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth,prediksi_perusahaan,prediksi_posisi,cosine_similarity,relevan/tidak,precision@5
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"[Perawat, Nurse, Medical Representative]","[Serrebeauty, PT Transfarma Medica Indah, Nusa...","[Dokter Umum, Medical Representative (Surabaya...","[0.7163448333740234, 0.6829164028167725, 0.672...","[1, 1, 0, 0, 1]",0.6
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","[Perawat, Nurse, Medical Representative]","[Klinik Sakti Medika, EUROMEDICA GROUP, PT RAD...","[Perawat Bedah, perawat, Nurse - Perawat, Nurs...","[0.5552957057952881, 0.5384441614151001, 0.527...","[1, 1, 1, 1, 0]",0.8
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"[Customer Service, Customer Success Officer, C...","[PT Cilia Prisma Utama Makmur, PT Giant Transp...",[Sales Representative Tangerang / Jakarta / Be...,"[0.6683275699615479, 0.6631754636764526, 0.662...","[0, 1, 1, 0, 1]",0.6
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",[Data Analyst for Production Staff (CODE : PRO...,"[PT Panasonic Gobel Energy Indonesia (PECGI), ...",[Data Analyst for Production Staff (CODE : PRO...,"[0.600621223449707, 0.5836132764816284, 0.5434...","[1, 0, 0, 1, 0]",0.4
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"[Engineer Staff, Project Engineer, Electrical ...","[PT Derma Enom Farma, PT. Sarana Meditama Inte...","[Electrical Engineering (Kosmetik & Farmasi), ...","[0.6778924465179443, 0.6648625135421753, 0.628...","[1, 0, 1, 0, 0]",0.4


Hitung rata-rata precision@5

In [42]:
mean_precision_at_5_embedding = df_cv_hasil_embedding['precision@5'].mean()
print(f"Nilai rata-rata precision@5: {mean_precision_at_5_embedding}")

Nilai rata-rata precision@5: 0.282


# **Embedding (with_punctuation)**

## Model

Kolom "text_for_embed_with_punctuation" dari dataset job dijadikan list

In [43]:
job_texts = df_job["text_for_embed_with_punctuation"].tolist()

Memanggil model embedding pre-train untuk prediksi

In [44]:
model = SentenceTransformer('all-MiniLM-L6-v2')
# model = SentenceTransformer('all-mpnet-base-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [45]:
job_embeddings = model.encode(job_texts, show_progress_bar=True)

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Mulai prediksi berdasarkan kolom "text_for_embed_with_punctuation" dari **df_cv** dan **df_job**

In [46]:
prediksi_perusahaan = []
prediksi_posisi = []
hasil_cosine_similarity = []

for h in range(len(df_cv)):
    prediksi_perusahaan_tiap_baris = []
    prediksi_posisi_tiap_baris = []
    hasil_cosine_similarity_tiap_baris = []

    # print("-"*40)
    # print(f"BARIS {h+1}")
    # print("-"*40)
    cv_text = df_cv.loc[h, 'text_for_embed_with_punctuation']
    cv_embedding = model.encode([cv_text])

    scores_embed = cosine_similarity(cv_embedding, job_embeddings)

    top_k = 5
    top_indices = scores_embed[0].argsort()[-top_k:][::-1]

    for i in top_indices:
        # print("Perusahaan:", df_job.iloc[i]["Perusahaan"])
        # print("Posisi:", df_job.iloc[i]["Posisi"])
        # print("Score:", scores_embed[0][i])
        # print("-"*40)
        prediksi_perusahaan_tiap_baris.append(df_job.iloc[i]["Perusahaan"])
        prediksi_posisi_tiap_baris.append(df_job.iloc[i]["Posisi"])
        hasil_cosine_similarity_tiap_baris.append(float(scores_embed[0][i]))

    prediksi_perusahaan.append(prediksi_perusahaan_tiap_baris)
    prediksi_posisi.append(prediksi_posisi_tiap_baris)
    hasil_cosine_similarity.append(hasil_cosine_similarity_tiap_baris)

    # print("\n")

Membuat salinan dari **df_cv** dan menambahkan beberapa kolom yang berisi hasil prediksi

In [47]:
df_cv_hasil_embedding_punctuation = df_cv.copy()
df_cv_hasil_embedding_punctuation['prediksi_perusahaan'] = prediksi_perusahaan
df_cv_hasil_embedding_punctuation['prediksi_posisi'] = prediksi_posisi
df_cv_hasil_embedding_punctuation['cosine_similarity'] = hasil_cosine_similarity
df_cv_hasil_embedding_punctuation.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth,prediksi_perusahaan,prediksi_posisi,cosine_similarity
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"[Perawat, Nurse, Medical Representative]","[Serrebeauty, PT Transfarma Medica Indah, PT B...","[Dokter Umum, Medical Representative (Surabaya...","[0.71684730052948, 0.6527622938156128, 0.63205..."
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","[Perawat, Nurse, Medical Representative]","[MAYAPADA HEALTHCARE, Klinik Sakti Medika, EUR...","[HEAD OF MAYAPADA CLINIC, Perawat Bedah, peraw...","[0.5217645764350891, 0.5140619874000549, 0.507..."
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"[Customer Service, Customer Success Officer, C...","[PT.Bumi Lancang kuning Pusaka (BLKP), PT Aldm...","[Customer Relation, Customer Service, Call Cen...","[0.6257870197296143, 0.6255267858505249, 0.618..."
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",[Data Analyst for Production Staff (CODE : PRO...,"[PT. HONGXIN ALGAE INTERNATIONAL, PT Sesawi Be...","[Production Supervisor, QC Analyst, Operationa...","[0.5989642143249512, 0.5543785095214844, 0.548..."
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"[Engineer Staff, Project Engineer, Electrical ...",[PT. Sarana Meditama International (EMC Health...,"[AI Application Officer, Mechanical and Electr...","[0.6134305596351624, 0.589758038520813, 0.5865..."


## Evaluasi

Menyiapkan preprocessing sederhana dan model pre-train untuk evaluasi

In [48]:
def normalize(text):
    text = text.lower()
    text = re.sub(r'\([^)]*\)', '', text)  # hapus isi kurung
    return text.strip()

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Menghitung nilai cosine similarity maksimum antara **ground truth** dan **hasil prediksi** pada setiap data, kemudian menentukan **threshold similarity** berdasarkan nilai kuartil ketiga (Q3) dari distribusi similarity maksimum yang diperoleh.

In [49]:
nilai_sim_max = []

for i in range(len(df_cv_hasil_embedding_punctuation)):

    ground_truth = [normalize(x) for x in df_cv_hasil_embedding_punctuation.loc[i,'ground_truth']]
    hasil_prediksi = [normalize(x) for x in df_cv_hasil_embedding_punctuation.loc[i,'prediksi_posisi']]

    sim = cosine_similarity(
        model.encode(ground_truth),
        model.encode(hasil_prediksi)
    )
    nilai_sim_max.extend(sim.max(axis=0))

In [50]:
series_sim_max = pd.Series(nilai_sim_max)
q3 = series_sim_max.quantile(0.75)
threshold = np.floor(q3 * 10) / 10
print(f'q3 = {q3}')
print(f'threshold = {threshold}')

q3 = 0.6978838741779327
threshold = 0.6


Menentukan relevansi setiap hasil prediksi berdasarkan nilai cosine similarity terhadap **ground truth** menggunakan threshold yang telah diperoleh, kemudian menghitung nilai **Precision@5** untuk mengukur proporsi prediksi yang relevan pada setiap data.

In [51]:
relevan_atau_tidak = []
nilai_precision_at_5 = []

for i in range(len(df_cv_hasil_embedding_punctuation)):

    ground_truth = [normalize(x) for x in df_cv_hasil_embedding_punctuation.loc[i,'ground_truth']]
    hasil_prediksi = [normalize(x) for x in df_cv_hasil_embedding_punctuation.loc[i,'prediksi_posisi']]

    sim = cosine_similarity(
        model.encode(ground_truth),
        model.encode(hasil_prediksi)
    )

    relevan = (sim.max(axis=0) >= threshold).astype(int).tolist()
    precision_at_5 = sum(relevan) / len(relevan)

    relevan_atau_tidak.append(relevan)
    nilai_precision_at_5.append(precision_at_5)

df_cv_hasil_embedding_punctuation['relevan/tidak'] = relevan_atau_tidak
df_cv_hasil_embedding_punctuation['precision@5'] = nilai_precision_at_5
df_cv_hasil_embedding_punctuation.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth,prediksi_perusahaan,prediksi_posisi,cosine_similarity,relevan/tidak,precision@5
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"[Perawat, Nurse, Medical Representative]","[Serrebeauty, PT Transfarma Medica Indah, PT B...","[Dokter Umum, Medical Representative (Surabaya...","[0.71684730052948, 0.6527622938156128, 0.63205...","[1, 1, 1, 0, 1]",0.8
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","[Perawat, Nurse, Medical Representative]","[MAYAPADA HEALTHCARE, Klinik Sakti Medika, EUR...","[HEAD OF MAYAPADA CLINIC, Perawat Bedah, peraw...","[0.5217645764350891, 0.5140619874000549, 0.507...","[0, 1, 1, 1, 1]",0.8
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"[Customer Service, Customer Success Officer, C...","[PT.Bumi Lancang kuning Pusaka (BLKP), PT Aldm...","[Customer Relation, Customer Service, Call Cen...","[0.6257870197296143, 0.6255267858505249, 0.618...","[1, 1, 0, 1, 1]",0.8
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",[Data Analyst for Production Staff (CODE : PRO...,"[PT. HONGXIN ALGAE INTERNATIONAL, PT Sesawi Be...","[Production Supervisor, QC Analyst, Operationa...","[0.5989642143249512, 0.5543785095214844, 0.548...","[1, 0, 0, 0, 0]",0.2
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"[Engineer Staff, Project Engineer, Electrical ...",[PT. Sarana Meditama International (EMC Health...,"[AI Application Officer, Mechanical and Electr...","[0.6134305596351624, 0.589758038520813, 0.5865...","[0, 1, 0, 1, 0]",0.4


Hitung rata-rata precision@5

In [52]:
mean_precision_at_5_embedding_punctuation = df_cv_hasil_embedding_punctuation['precision@5'].mean()
print(f"Nilai rata-rata precision@5: {mean_precision_at_5_embedding_punctuation}")

Nilai rata-rata precision@5: 0.38000000000000006


# **Hasil Evaluasi Precision@5**

In [53]:
pd.DataFrame({
    'Model': ['TF-IDF', 'TF-IDF + SVD', 'Embedding', 'Embedding (with_punctuation)'],
    'Nilai rata-rata precision@5': [mean_precision_at_5_tfidf, mean_precision_at_5_tfidf_svd, mean_precision_at_5_embedding, mean_precision_at_5_embedding_punctuation]
})

,Model,Nilai rata-rata precision@5
0,TF-IDF,0.280
1,TF-IDF + SVD,0.334
2,Embedding,0.282
3,Embedding (with_punctuation),0.380
